In [ ]:
from pathlib  import Path
import pandas as pd
import matplotlib as plt
import os
import sys
import json
import nltk
from nltk import FreqDist
import matplotlib.pyplot as plt
from unidecode import unidecode
from gensim.parsing.preprocessing import split_alphanum
from gensim.parsing.preprocessing import strip_non_alphanum
from gensim.parsing.preprocessing import strip_punctuation
from gensim.parsing.preprocessing import strip_tags
from gensim.parsing.preprocessing import strip_multiple_whitespaces
sys.path.append('../')
import spacy
nlp = spacy.load("fr_core_news_sm")
import numpy as np

In [ ]:
path_name = '/home/ml/BI_Help/data/raw/Fusion_Procurement_BI/'
path_file_xls = '/home/ml/BI_Help/data/raw/Import/corresp_Labelle/text_material_group_AVS_FR.xlsx'
path_file_text = '/home/ml/BI_Help/data/raw/Import/corresp_Labelle/MATKL T023T ALL AQ6.txt'

In [ ]:
path_file_pkl ='/home/ml/BI_Help/data/processed/codes_labels.pkl'
path_code_corrige = '/home/ml/BI_Help/data/processed/codes_corrige.pkl'

In [ ]:
EKPO_files = sorted(Path(path_name).glob("*EKPO*"))

In [ ]:
def merge_total():
    df_list = []
    for i in range(len(EKPO_files)):
        df_list.append(pd.read_csv(str(EKPO_files[i]),sep = ';', error_bad_lines=False))
    return pd.concat(df_list)

def lemma(x):
    doc = nlp(x)
    tokens = [token.text for token in doc]
    list_test = []
    for token in doc:
        list_test.append(token.lemma_)
    return list_test

def correction(row):
    if 'utiliser' in str(row['Long']):
        return row['Long'].replace('utiliser ','').strip()
    else:
        return row['MATKLMaterialGroup']

In [ ]:
df_total = merge_total()

In [ ]:
df_correction = pd.read_pickle(path_code_corrige)

In [ ]:
df_correction

In [ ]:
df_total = df_total[['TXZ01 Short Text' , 'MATKL Material Group']]
df_total.drop_duplicates(inplace=True)

In [ ]:
df_correction.rename(columns = {'Matl Group' : 'MATKL Material Group'}, inplace=True)

In [ ]:
df_merge = pd.merge(df_total, df_correction, on='MATKL Material Group', how='left')

In [ ]:
def correction_test(row):
    
    if row.notnull()['code_correction']:
        return row['code_correction']
    else :
        return row['MATKL Material Group']

In [ ]:
df_merge['code_correction'] = df_merge[['MATKL Material Group', 'code_correction']].apply(correction_test, axis=1)

In [ ]:
def df_malk_correction():
    df_malk = pd.read_pickle(path_file_pkl)
    df_malk['Matl Group'] = df_malk['Matl Group'].astype(str)
    df_malk['Material grp desc. 2'] = df_malk['Material grp desc. 2'].astype(str)
    def correction(row):
        if 'utiliser' in str(row['Material grp desc. 2']):
            return row['Material grp desc. 2'].replace('utiliser ', '').strip()
        else:
            return row['Matl Group']
    df_malk['code_correction'] = df_malk[['Material grp desc. 2', 'Matl Group']].apply(correction, axis=1)
    df_malk.dropna(inplace=True)
    return df_malk[~df_malk['Material grp desc. 2'].str.contains("utiliser")]

In [ ]:
df_malk = df_malk_correction()

In [ ]:
df_util = pd.merge(df_merge, df_malk[['code_correction', 'Material grp desc. 2']], on = 'code_correction', how = 'left')

In [ ]:
df_util